# Chapter 1

### OOP in pytorch

```
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset): # This is a class that should behave like pytorch dataset
    def __init__(self, csv_path): # Constructor: should read dataset from pecified path
        super().__init__() # super constructor
        df = pd.read_csv(csv_path)
        self.data = df.to_numpy()
    def __len__(self): # Returns number of samples in dataset
        return self.data.shape[0]
    def __getitem__(self, idx): # returns features and labels for each given index
        features = self.data[idx, :-1]
        label = self.data[idx, -1]
        return features, label

dataset_train = WaterDataset("filename.csv")
dataloader_train = DataLoader(dataset_train, batch_size=2, shuffle=True)
features, labels = next(iter(dataloader_train))

# Creating a class based model
import torch.nn.init as init

class MyNet(nn.Module):
    def __init__(self): # constructor : should contain model layers
        super(Net, self).__init__()
        self.fc1 = nn.Linear(9, 16)
        self.bn1 = nn.BatchNorm1d(16) # Batch normalization layer
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        # Use proper weight initialization for each layer
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid") # indication that last layer will have sigmoid behavior


    def forward(self, x): # forward propagation : layers should be wrapped by activation with subsequent inputs as x
        x = self.fc1(x)
        x = self.bn1(x) # Passing the fc1 into batch normalization layer
        x = nn.functional.elu(x)
        x = self.fc2(x)
        x = nn.functional.relu(x)
        x = self.fc3(x)
        x = nn.functional.sigmoid(x)
        return x
net = MyNet()

import torch.nn as nn
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

net.train() # Set the model on training mode
for epoch in range(1000):
    for features, labels in dataloader_train:
        optimizer.zero_grad() # Reset grads before back-propagation
        outputs = net(features)
        loss = criterion(outputs, labels.view(-1, 1)) # reshaping torch labels
        loss.backward() # perform back-propagation
        optimizer.step() # perform optimization


from torchmetrics import Accuracy
acc = Accuracy(task="binary")
net.eval() # Set the model on evaluation mode 

with torch.no_grad(): # No gradient calculation during evaluation
    for features, labels in dataloader_test:
        outputs = net(features)
        preds = (outputs >= 0.5).float() # prediction values converted to 0 or >= 0.5 values
        acc(preds, labels.view(-1, 1)) # This is just appending the values. the real calculation is not done yet

accuracy = acc.compute() # Perform the calculation

```

### Dying Neurons, Exploding gradients, Vanishing gradients

<center><img src="images/01.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.03.png"  style="width: 400px, height: 300px;"/></center>

- - gradients represent the slope of the loss function with respect to the model parameters of a neural network
- Duying Neuron Problem:
	- This happens when a neuron takes a value less than 0 for all the inputs
    - When relu is used and when the node val is 0, 
    	updated weight = current weight - node val * slope
        => updated weight = current weight (no change,  neuron comes to a standstill)
	- Occurs when a neuron keeps getting negative inputs
    - Solution: Use different activation function other than relu 
        - eg: elu, Non-zero gradients for negative values that overcomes dying neuron problem, and also average output around zero - helps against vanishing gradient problem
- Vanishing Gradients:
	- Prevents neural networks from booming sooner (when updates to weights 
		during backpropagation are close to 0)
	- Happens when sigmoid or softmax activation function is used
    - For high and low value of x, the gradients approach to 0 (saturation behavior)
	- Occurs when many layers of neural network have very small slopes (e.g. due to
		being on the flat part of the tangent curve)
	- Sigmoid function makes the intermediate values of the neural network 
		between 0 and 1 
	- When back-propagation is used, we keep multiplying values less 
		than 1 with each other. 
	- So the gradient keeps getting smaller and smaller moving 
		backward to the network.
	- So, the neurons in the earlier layers learn slowly 
		compared to the neurons in the later layers in the network
	- Thus, training takes too long and accuracy is compromised.
    - Use RELU activation : gradients do not converge to 0 for high value of x
    - Use Leaky RELU activation :For negative inputs, it multiplies the input by a small coefficient (defaulted to 0.01)
    	- For this, gradients for negative inputs are never null for leaky relu
- Exploding Gradients:
	- Opposite to vanishing gradients
    - Gradients get bigger and bigger
    - Parameter updates are too large
    - Training diverges
	- Happens when a batch of data is totally different from rest of the data
    - For high and low value of x, the gradients approach to 0 (saturation behavior)
	- Occurs when many layers of neural network have very small slopes (e.g. due to
		being on the flat part of the tangent curve
	- Sigmoid function makes the intermediate values of the neural network 
		between 0 and 1 
	- When back-propagation is used, we keep multiplying values less 
		than 1 with each other. 
	- So the gradient keeps getting smaller and smaller moving 
		backward to the network.
	- So, the neurons in the earlier layers learn slowly 
		compared to the neurons in the later layers in the network
	- Thus, training takes too long and accuracy is compromised.
    - Use RELU activation : gradients do not converge to 0 for high value of x
    - Use Leaky RELU activation :For negative inputs, it multiplies the input by a small coefficient (defaulted to 0.01)
    	- For this, gradients for negative inputs are never null for leaky relu
- Best practices for regularization:
    - Use good activation functions (eg: ELU or leaky relu)
    - Use batch normalization 
		- Batch normalization effectively learns the optimal input distribution for each layer it precedes. By learning how to optimally re-scale the next layer's inputs, batch normalization mitigates the unstable gradients problems!
    - Use proper weight initialization (eg: For ReLU and similar, we can use He/Kaiming initialization)